In [15]:
import torch
from transformers import AutoModelForCausalLM, AutoProcessor, AutoModelForMultimodalLM, Qwen2_5_VLForConditionalGeneration, BitsAndBytesConfig, Qwen2VLForConditionalGeneration
from datasets import Dataset, Image, load_dataset
from qwen_vl_utils import process_vision_info
import json

In [ ]:
model = Qwen2VLForConditionalGeneration.from_pretrained(
    "Qwen/Qwen2-VL-2B-Instruct", torch_dtype="auto", device_map="auto"
)

In [ ]:
processor = AutoProcessor.from_pretrained("Qwen/Qwen2-VL-2B-Instruct")

In [ ]:
from qwen_vl_utils import pro

In [30]:
def field_accuracy(prediction, target):
    correct = 0
    prediction = json.loads(prediction)
    target = json.loads(target)
    for field, target_value in target.items():
        pred_value = prediction.get(field).strip().lower()

        if pred_value == target_value.lower():
            correct += 1

    return correct / len(target)

In [36]:
ds = load_dataset("mychen76/ds_receipts_v2_test", split="train[-100:]")

In [32]:
EXTRACTION_PROMPT = """
Извлеки информацию с изображения чека.

Верни результат строго в формате JSON следующей структуры:

{
    "store_name": "",
    "store_addr": "",
    "telephone": "",
    "date": "",
    "time": "",
    "total": "",
    "line_items": [
        {
            "item_name": "",
            "item_value": "",
            "item_quantity": ""
        }
    ]
}

Не добавляй никакого дополнительного текста до или после JSON.
Если значение отсутствует на изображении, оставь поле пустым.
"""

In [41]:
def _make_messages(sample):
  messages = [
        {
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "image": sample['image']
                },
                {
                    "type": "text",
                    "text": EXTRACTION_PROMPT
            }]}
            
  ]

  return {'messages': messages}

In [16]:
def _target_change(t: str):
    t = json.loads(json.loads(t))
    del t['tax']
    del t['subtotal']
    del t['ignore']
    del t['tips']
    for d in t['line_items']:
        del d['item_key']
    return json.dumps(t, ensure_ascii=False)


In [18]:
import pandas as pd

In [37]:
ds = ds.to_pandas()
ds['text'] = ds['text'].map(_target_change)
ds = Dataset.from_pandas(ds).cast_column('image', Image())

In [38]:
json.loads(ds['text'][0])

{'store_name': 'RedRobinGourmetBurgers',
 'store_addr': '14090WorthAve Woodbridge,VA22192',
 'telephone': '703-492-6900',
 'date': '05/20/2016',
 'time': '9:57PM',
 'total': '',
 'line_items': [{'item_name': 'PORTCITYOPTIMALKIT',
   'item_value': '5.49',
   'item_quantity': '1'},
  {'item_name': 'BLEUBG', 'item_value': '10.69', 'item_quantity': '1'},
  {'item_name': '+MUSHRMS', 'item_value': '1.39', 'item_quantity': '1'},
  {'item_name': 'GNTDOGFISHHEAD60IPA',
   'item_value': '7.79',
   'item_quantity': '1'},
  {'item_name': 'MUSHBG', 'item_value': '5.14', 'item_quantity': '1'}]}

In [42]:
messages = [_make_messages(example) for example in ds]

In [44]:
from tqdm.auto import tqdm
progress_bar = tqdm(range(len(messages)))

  0%|          | 0/100 [00:00<?, ?it/s]

In [ ]:
for message in messages:
    text = processor.apply_chat_template(
        message, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(message)
    inputs = processor(
        text=[text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    )
    inputs = inputs.to("cuda")

    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=256)
    generated_ids_trimmed = [
        out_ids[len(in_ids) :] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)
    ]
    output_text = processor.batch_decode(
        generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    progress_bar.update(1)

In [45]:
c = [1,2,3]
g = [6, 5, 4]

In [48]:
c = [0.1, 0.231]
print(sum(c))

0.331
